In [12]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
df = pd.read_csv(r'../data/results.csv')

df['date'] = pd.to_datetime(df['date'])
df = df[df['tournament'] != 'Friendly']

def winner(row):
    if(row['home_score'] > row['away_score']):
        return 'home'
    elif(row['home_score'] < row['away_score']):
        return 'away'
    else:
        return ('draw')

df['winner'] = df.apply(winner, axis=1)

home_df = pd.DataFrame({
    'date': df['date'],
    'team': df['home_team'],
    'won': df['winner'] == 'home',
    'goals_scored': df['home_score'],
    'goals_conceded': df['away_score']
})

away_df = pd.DataFrame({
    'date': df['date'],
    'team': df['away_team'],
    'won': df['winner'] == 'away',
    'goals_scored': df['away_score'],
    'goals_conceded': df['home_score']
})

win_result = pd.concat([home_df, away_df])
win_result = win_result.reset_index(drop=True)

win_rate = win_result.groupby('team')['won'].mean()
date_sorted = win_result.sort_values('date')
last_10 = date_sorted.groupby('team').tail(n=10)
win_rate_10 = last_10.groupby('team')['won'].mean()


goals = win_result.groupby('team')['goals_scored'].mean()
goals_allowed = win_result.groupby('team')['goals_conceded'].mean()
goal_differential = goals - goals_allowed


team_features = pd.concat([win_rate, win_rate_10, goals, goals_allowed, goal_differential], axis=1)
team_features.columns = ['overall_win_rate', 'recent_win_rate', 'goals_scored', 'goals_conceded', 'goal_differential']




def head_to_head(team1, team2):
    h2h = df[((df['home_team'] == team1) & (df['away_team'] == team2)) | ((df['away_team'] == team1) & (df['home_team'] == team2))]
    if(len(h2h) == 0):
        return 0.5
    team1_wins = len(h2h[((h2h['home_team'] == team1) & (h2h['winner'] == 'home')) |
                         ((h2h['away_team'] == team1) & (h2h['winner'] == 'away'))])
    return team1_wins / len(h2h)




match_data = df.merge(team_features, left_on='home_team', right_index=True)
match_data = match_data.merge(team_features, left_on='away_team', right_index=True, suffixes=('_home', '_away'))
print(match_data.columns)

def result(row):
    if(row['home_score'] > row['away_score']):
        return 1
    elif(row['home_score'] < row['away_score']):
        return -1
    else:
        return 0

match_data['result'] = match_data.apply(result, axis=1)
print(match_data[['home_team', 'away_team', 'winner', 'result']].head())

feature_cols = ['overall_win_rate_home', 'recent_win_rate_home', 'goals_scored_home', 'goals_conceded_home', 'goal_differential_home', 'overall_win_rate_away', 'recent_win_rate_away', 'goals_scored_away', 'goals_conceded_away', 'goal_differential_away']
X = match_data[feature_cols]
y = match_data['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(len(X_train))
print(len(X_test))

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)


Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral', 'winner',
       'overall_win_rate_home', 'recent_win_rate_home', 'goals_scored_home',
       'goals_conceded_home', 'goal_differential_home',
       'overall_win_rate_away', 'recent_win_rate_away', 'goals_scored_away',
       'goals_conceded_away', 'goal_differential_away'],
      dtype='str')
           home_team         away_team winner  result
29  Northern Ireland          Scotland   away      -1
30             Wales  Northern Ireland   home       1
31  Northern Ireland           England   away      -1
32          Scotland           England   home       1
33             Wales           England   away      -1
24712
6178
0.5524441566850113
